In [32]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines


In [33]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

MASTER_PATH = Path("../../analysis/brains/ogse_experiments/master.long.parquet")
OUT_DIR     = Path("../../analysis/brains/ogse_experiments/lab/quick_plotting")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DIRECTIONS = ["tra", "long"]
N_LIST     = [1, 4, 8]
TD_LIST    = [76.0, 120.0, 143.4, 210.0]
# ROI_LIST
#    None → All ROIS
#    ["AntCC","MidAntCC","CentralCC","MidPostCC","PostCC"] → CC
#    ["Left-Lateral-Ventricle", "Right-Lateral-Ventricle"] → Ventricles
ROI_LIST    = ["AntCC","MidAntCC","CentralCC","MidPostCC","PostCC","Left-Lateral-Ventricle", "Right-Lateral-Ventricle"]
G_COLUMN    = "g_thorsten"
G_CORRECTION_COLUMN = "grad_correction_factor"
Y_COLUMN    = "value"   # "value" (raw a.u.) or "value_norm" (normalized by S0)

gamma = 267.5221900 

# ── Direction subtraction: diff = signal(DIFF_DIR_A) − signal(DIFF_DIR_B) ──
DIFF_DIR_A = "tra"
DIFF_DIR_B = "long"

# ── Rician noise floor RN (signal units) — same resolution logic as fit_ogse_tort.ipynb ──
# Normalized plots divide the signal by M0 = sqrt(S0² − RN²).
#   None                        → no noise correction (RN = 0)
#   float                       → same RN for all subjects, td, and roi
#   {td: float}                 → fixed per td, same for all subjects and roi
#   {subj: float}               → fixed per subject, same for all td and roi
#   {subj: {td: float}}         → fixed per subject and td, same for all roi
#   {subj: {td: {roi: float}}}  → fixed per subject, td, and roi
RN_FIXED = {
    "BRAIN": {
        76.0:  {"AntCC": 37, "MidAntCC": 38, "CentralCC": 48, "MidPostCC": 49, "PostCC": 53},
        90.0:  {"AntCC": 32, "MidAntCC": 38, "CentralCC": 36, "MidPostCC": 42, "PostCC": 36},
        120.0: {"AntCC": 36, "MidAntCC": 44, "CentralCC": 40, "MidPostCC": 46, "PostCC": 40},
        143.4: {"AntCC": 36, "MidAntCC": 44, "CentralCC": 40, "MidPostCC": 46, "PostCC": 40},
        210.0: {"AntCC": 37, "MidAntCC": 38, "CentralCC": 48, "MidPostCC": 49, "PostCC": 53},
    },
    "LUDG": {
        90.0:  {"AntCC": 31, "MidAntCC": 35, "CentralCC": 38, "MidPostCC": 48, "PostCC": 49},
        120.0: {"AntCC": 30, "MidAntCC": 42, "CentralCC": 34, "MidPostCC": 44, "PostCC": 44},
        143.4: {"AntCC": 30, "MidAntCC": 42, "CentralCC": 34, "MidPostCC": 44, "PostCC": 44},
        210.0: {"AntCC": 31, "MidAntCC": 35, "CentralCC": 38, "MidPostCC": 48, "PostCC": 49},
    },
    "MBBL": {
        90.0:  {"AntCC": 35, "MidAntCC": 40, "CentralCC": 40, "MidPostCC": 52, "PostCC": 51},
        120.0: {"AntCC": 31, "MidAntCC": 41, "CentralCC": 47, "MidPostCC": 49, "PostCC": 44},
        143.4: {"AntCC": 31, "MidAntCC": 41, "CentralCC": 47, "MidPostCC": 49, "PostCC": 44},
        210.0: {"AntCC": 35, "MidAntCC": 40, "CentralCC": 40, "MidPostCC": 52, "PostCC": 51},
    },
}


In [34]:
# ══════════════════════════════════════════════════════════════════════════════
# LOAD DATA
# ══════════════════════════════════════════════════════════════════════════════

df = pd.read_parquet(MASTER_PATH)

data = df[
    (df.row_kind == "signal_rotated") &
    (df.direction.isin(DIRECTIONS)) &
    (df.N.isin(N_LIST)) &
    (df.stat == "avg")
].copy()

missing_cols = [c for c in [G_COLUMN, Y_COLUMN, G_CORRECTION_COLUMN] if c is not None and c not in data.columns]
if missing_cols:
    raise KeyError(f"Missing column(s): {missing_cols}")

data["G_fit"] = data[G_COLUMN]
if G_CORRECTION_COLUMN is not None:
    data["G_fit"] = data["G_fit"] * data[G_CORRECTION_COLUMN]
data["y_raw"] = data["value"]   # raw signal — always kept to derive S0
data["y_fit"] = data[Y_COLUMN]

G_LABEL = f"$b_{{value}} = \\frac{{\gamma^2 G^2 T_d^3}}{{12 N^2}}$ "  #f"Modulation gradient G [mT/m] ({G_COLUMN})"
Y_LABEL = f"{Y_COLUMN} [a.u.]"

if TD_LIST is not None:
    data = data[data.td_ms.isin(TD_LIST)]
if ROI_LIST is not None:
    data = data[data.roi.isin(ROI_LIST)]

print(f"Rows loaded: {len(data)}")
print("Groups (subj, roi, dir):", data.groupby(["subj", "roi", "direction"]).ngroups)
data[["subj", "roi", "direction", "td_ms", "N", "G_fit", "y_fit"]]


Rows loaded: 1804
Groups (subj, roi, dir): 14


,subj,roi,direction,td_ms,N,G_fit,y_fit
25784,BRAIN,AntCC,tra,120.0,1,0.000000,126.561067
25785,BRAIN,AntCC,tra,120.0,1,1.393733,114.366234
25786,BRAIN,AntCC,tra,120.0,1,2.787466,98.752041
25787,BRAIN,AntCC,tra,120.0,1,4.181199,103.868817
25788,BRAIN,AntCC,tra,120.0,1,5.574932,100.994628
...,...,...,...,...,...,...,...
72639,BRAIN,Right-Lateral-Ventricle,long,210.0,8,35.082040,62.933029
72640,BRAIN,Right-Lateral-Ventricle,long,210.0,8,40.929046,47.414143
72641,BRAIN,Right-Lateral-Ventricle,long,210.0,8,46.776053,47.191105
72642,BRAIN,Right-Lateral-Ventricle,long,210.0,8,52.623060,46.266957


In [35]:
# ══════════════════════════════════════════════════════════════════════════════
# RN / M0 HELPERS  (same resolution logic as fit_ogse_tort.ipynb)
# ══════════════════════════════════════════════════════════════════════════════

def _resolve_rn(subj, td, roi=None):
    # Return fixed RN for (subj, td, roi). Returns 0.0 when RN_FIXED is None.
    if RN_FIXED is None:
        return 0.0
    if isinstance(RN_FIXED, dict):
        first_key = next(iter(RN_FIXED))
        if isinstance(first_key, str):               # {subj: ...}
            subj_entry = RN_FIXED.get(subj)
            if subj_entry is None:
                return 0.0
            if isinstance(subj_entry, dict):         # {subj: {td: ...}}
                td_entry = subj_entry.get(float(td), subj_entry.get(td))
                if td_entry is None:
                    return 0.0
                if isinstance(td_entry, dict):       # {subj: {td: {roi: float}}}
                    return float(td_entry.get(roi, 0.0))
                return float(td_entry)               # {subj: {td: float}}
            return float(subj_entry)                 # {subj: float}
        return float(RN_FIXED.get(float(td), RN_FIXED.get(td, 0.0)))  # {td: float}
    return float(RN_FIXED)


def _m0_from_s0(s0_raw, rn):
    # M0 = sqrt(S0² − RN²). For value_norm returns M0/S0.
    m0 = np.sqrt(max(float(s0_raw)**2 - float(rn)**2, 0.0))
    if Y_COLUMN == "value_norm":
        return m0 / float(s0_raw) if float(s0_raw) > 0 else 1.0
    return m0


In [36]:
# ══════════════════════════════════════════════════════════════════════════════
# BUILD DATA STORE — one entry per (subj, roi, direction), curves per (td, N)
# ══════════════════════════════════════════════════════════════════════════════

data_store = {}

for (subj, roi, direction), grp in data.groupby(["subj", "roi", "direction"]):
    tds = sorted(float(t) for t in grp.td_ms.unique())
    data_store[(subj, roi, direction)] = {"tds": tds, "curves": {td: {} for td in tds}}

    for td in tds:
        sub = grp[np.isclose(grp.td_ms.astype(float), td)]
        for N in N_LIST:
            s = sub[sub.N == N].sort_values("G_fit")
            if s.empty:
                continue
            G = s.G_fit.values
            y = s.y_fit.values
            s0_raw = float(s.loc[s.G_fit.abs() == s.G_fit.abs().min(), "y_raw"].iloc[0])
            rn = _resolve_rn(subj, td, roi)
            M0 = _m0_from_s0(s0_raw, rn)
            y_norm = (y**2 - rn**2) / M0**2 if M0 != 0.0 else np.full_like(y, np.nan)
            data_store[(subj, roi, direction)]["curves"][td][N] = dict(
                G= ( gamma**2 * G**2 * td**3 ) / (12 * N**2), y=y, y_norm=y_norm, S0=s0_raw, RN=rn, M0=M0,
            )

print("Groups (subj, roi, dir):", len(data_store))


Groups (subj, roi, dir): 14


In [37]:
# ══════════════════════════════════════════════════════════════════════════════
# BUILD DIFF DATA STORE — signal(DIFF_DIR_A) − signal(DIFF_DIR_B), per (subj, roi)
# ══════════════════════════════════════════════════════════════════════════════
# G_fit differs slightly between directions (grad_correction_factor is per-direction),
# so curves are aligned by position along the sorted gradient-step sequence rather
# than by exact G value; DIFF_DIR_A's G values are kept for the x-axis.

data_store_diff = {}

for (subj, roi, direction), store_a in data_store.items():
    if direction != DIFF_DIR_A:
        continue
    store_b = data_store.get((subj, roi, DIFF_DIR_B))
    if store_b is None:
        continue

    tds = sorted(set(store_a["tds"]) & set(store_b["tds"]))
    curves = {td: {} for td in tds}

    for td in tds:
        curves_a = store_a["curves"][td]
        curves_b = store_b["curves"][td]
        for N in N_LIST:
            ca = curves_a.get(N)
            cb = curves_b.get(N)
            if ca is None or cb is None:
                continue
            if len(ca["G"]) != len(cb["G"]):
                print(f"Skipping {subj}/{roi}/td={td}/N={N}: different number of gradient steps between {DIFF_DIR_A} and {DIFF_DIR_B}")
                continue
            curves[td][N] = dict(
                G=ca["G"],
                y=ca["y"] - cb["y"],
                y_norm=ca["y_norm"] - cb["y_norm"],
            )

    data_store_diff[(subj, roi)] = {"tds": tds, "curves": curves}

print("Groups (subj, roi) with diff:", len(data_store_diff))


Groups (subj, roi) with diff: 7


In [38]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT SIGNAL — grid: one subplot per td, one curve per N
# ══════════════════════════════════════════════════════════════════════════════

N_colors = {N: c for N, c in zip(N_LIST, ["C0", "C1", "C2", "C3"])}

def _plot_grid(value_key, ylabel, out_prefix, store_dict=None):
    store_dict = data_store if store_dict is None else store_dict
    for key, store in store_dict.items():
        subj, roi, direction = key if len(key) == 3 else (*key, f"{DIFF_DIR_A}-{DIFF_DIR_B}")
        tds = store["tds"]
        n_tds = len(tds)
        fig, axes = plt.subplots(1, n_tds, figsize=(4 * n_tds, 3.5), sharey=True, squeeze=False)
        axes = axes[0]

        for ax, td in zip(axes, tds):
            curves_td = store["curves"][td]
            for N in N_LIST:
                curve = curves_td.get(N)
                if curve is None:
                    continue
                ax.scatter(curve["G"], curve[value_key], color=N_colors[N], s=20, zorder=3, label=f"N={N}")
                ax.plot(curve["G"], curve[value_key], color=N_colors[N], linewidth=1)
            ax.set_title(f"td = {td:.1f} ms", fontsize=8)
            ax.set_xlabel(G_LABEL)
            ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
            ax.set_axisbelow(True)

        axes[0].set_ylabel(ylabel)
        axes[0].legend(fontsize=6)
        axes[0].set_yscale('log')
        fig.suptitle(f"{subj}  |  {roi}  |  {direction}", fontsize=9)
        fig.tight_layout()
        fig.savefig(OUT_DIR / f"{out_prefix}_{subj}_{roi}_{direction}.png", dpi=120)
        plt.close(fig)

_plot_grid("y", Y_LABEL, "signal_raw")
print("Raw signal grid plots saved.")


Raw signal grid plots saved.


In [39]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT NORMALIZED SIGNAL — grid: one subplot per td, one curve per N
# Signal divided by M0 = sqrt(S0² − RN²)
# ══════════════════════════════════════════════════════════════════════════════

_plot_grid("y_norm", f"{Y_COLUMN} / M0", "signal_norm")
print("Normalized signal grid plots saved.")


Normalized signal grid plots saved.


In [40]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT COMBINED — single axes per (subj, roi, direction): all N × all td
# color = td, linestyle = N
# ══════════════════════════════════════════════════════════════════════════════

_N_LS = ["-", "--", "-.", ":"]
_TD_COLORS = ["C0", "C1", "C2", "C3", "C4", "C5", "C6", "C7"]

def _plot_combined(value_key, ylabel, out_prefix, store_dict=None):
    store_dict = data_store if store_dict is None else store_dict
    for key, store in store_dict.items():
        subj, roi, direction = key if len(key) == 3 else (*key, f"{DIFF_DIR_A}-{DIFF_DIR_B}")
        tds = store["tds"]
        td_colors = {td: _TD_COLORS[i % len(_TD_COLORS)] for i, td in enumerate(tds)}
        N_linestyles = {N: _N_LS[i % len(_N_LS)] for i, N in enumerate(N_LIST)}

        fig, ax = plt.subplots(figsize=(6, 5))
        for td in tds:
            curves_td = store["curves"][td]
            for N in N_LIST:
                curve = curves_td.get(N)
                if curve is None:
                    continue
                ax.scatter(curve["G"], curve[value_key], color=td_colors[td], s=15, zorder=3)
                ax.plot(curve["G"], curve[value_key], color=td_colors[td],
                        linestyle=N_linestyles[N], linewidth=1.2)

        ax.set_xlabel(G_LABEL)
        ax.set_ylabel(ylabel)
        ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
        ax.set_axisbelow(True)
        ax.set_yscale('log')

        td_handles = [mlines.Line2D([], [], color=td_colors[td], marker="o", linestyle="-", label=f"td={td:.1f} ms")
                      for td in tds]
        N_handles = [mlines.Line2D([], [], color="gray", linestyle=N_linestyles[N], label=f"N={N}")
                     for N in N_LIST]
        legend_td = ax.legend(handles=td_handles, fontsize=6, loc="upper right", title="td", title_fontsize=6)
        ax.add_artist(legend_td)
        ax.legend(handles=N_handles, fontsize=6, loc="lower right", title="N", title_fontsize=6)

        fig.suptitle(f"{subj}  |  {roi}  |  {direction}  —  combined", fontsize=9)
        fig.tight_layout()
        fig.savefig(OUT_DIR / f"{out_prefix}_{subj}_{roi}_{direction}.png", dpi=120)
        plt.close(fig)

_plot_combined("y", Y_LABEL, "signal_raw_combined")
print("Raw combined plots saved.")


Raw combined plots saved.


In [41]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT COMBINED — normalized signal, single axes per (subj, roi, direction)
# ══════════════════════════════════════════════════════════════════════════════

_plot_combined("y_norm", f"{Y_COLUMN} / M0", "signal_norm_combined")
print("Normalized combined plots saved.")


Normalized combined plots saved.


In [42]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT DIFF SIGNAL — grid: one subplot per td, one curve per N
# diff = signal(DIFF_DIR_A) − signal(DIFF_DIR_B)
# ══════════════════════════════════════════════════════════════════════════════

_plot_grid("y", f"{Y_LABEL} ({DIFF_DIR_A} − {DIFF_DIR_B})", "signal_raw_diff", store_dict=data_store_diff)
print("Raw diff signal grid plots saved.")


Raw diff signal grid plots saved.


In [43]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT DIFF NORMALIZED SIGNAL — grid: one subplot per td, one curve per N
# diff = signal(DIFF_DIR_A) − signal(DIFF_DIR_B), each normalized by its own M0
# ══════════════════════════════════════════════════════════════════════════════

_plot_grid("y_norm", f"{Y_COLUMN} / M0 ({DIFF_DIR_A} − {DIFF_DIR_B})", "signal_norm_diff", store_dict=data_store_diff)
print("Normalized diff signal grid plots saved.")


Normalized diff signal grid plots saved.


In [44]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT DIFF COMBINED — single axes per (subj, roi): all N × all td
# diff = signal(DIFF_DIR_A) − signal(DIFF_DIR_B)
# ══════════════════════════════════════════════════════════════════════════════

_plot_combined("y", f"{Y_LABEL} ({DIFF_DIR_A} − {DIFF_DIR_B})", "signal_raw_combined_diff", store_dict=data_store_diff)
print("Raw diff combined plots saved.")


Raw diff combined plots saved.


In [45]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT DIFF COMBINED — normalized signal, single axes per (subj, roi)
# ══════════════════════════════════════════════════════════════════════════════

_plot_combined("y_norm", f"{Y_COLUMN} / M0 ({DIFF_DIR_A} − {DIFF_DIR_B})", "signal_norm_combined_diff", store_dict=data_store_diff)
print("Normalized diff combined plots saved.")


Normalized diff combined plots saved.
